In [1]:
import pickle as pkl
from rdkit import Chem
from rdkit.Chem import Draw
import json
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm

In [2]:
scaffold_train = pkl.load(open("data/scaffold/drugs/scaffold_train_smiles.pkl", "rb"))
scaffold_val = pkl.load(open("data/scaffold/drugs/scaffold_val_smiles.pkl", "rb"))
scaffold_test = pkl.load(open("data/scaffold/drugs/scaffold_test_smiles.pkl", "rb"))

drugs_summary_json = json.load(open("data/rdkit/summary_drugs.json"))

In [3]:
len(scaffold_train), len(scaffold_val), len(scaffold_test)

(260752, 29185, 1000)

In [4]:
def get_canonical_smiles(smi: str) -> str:
    mol = Chem.MolFromSmiles(smi)
    [a.SetAtomMapNum(0) for a in mol.GetAtoms()]
    return Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )

In [5]:
print(get_canonical_smiles(scaffold_train[0]))

BrC(/C=C\c1ccccc1)=N\Nc1nc(N2CCOCC2)nc(N2CCOCC2)n1


In [12]:
from pathlib import Path
import pickle as pkl
from rdkit import Chem
import traceback

geom_drugs_prefix = Path("data/rdkit/")

def process_one(item):
    smi, data = item
    try:
        mol_dict = pkl.load(open(geom_drugs_prefix / data["pickle_path"], "rb"))
        confs = mol_dict["conformers"]

        canon_iso = None
        canon_noiso = None

        for conf in confs:
            mol = conf["rd_mol"]

            Chem.SanitizeMol(mol)
            mol = Chem.RemoveHs(mol)
            Chem.SanitizeMol(mol)

            iso = Chem.MolToSmiles(
                mol,
                canonical=True,
                isomericSmiles=True,
            )
            noiso = Chem.MolToSmiles(
                mol,
                canonical=True,
                isomericSmiles=False,
            )

            if "." in iso:
                return None, {
                    "smi": smi,
                    "reason": "disconnected_smiles",
                    "value": iso,
                }

            if canon_iso is None:
                canon_iso = iso
                canon_noiso = noiso
            elif iso != canon_iso:
                return None, {
                    "smi": smi,
                    "reason": "inconsistent_conformers",
                    "canon": canon_iso,
                    "other": iso,
                }

        # SUCCESS
        return smi, {
            "iso": canon_iso,
            "noiso": canon_noiso,
        }

    except Exception:
        return None, {
            "smi": smi,
            "reason": "exception",
            "traceback": traceback.format_exc(),
        }


In [13]:
from multiprocessing import Pool, cpu_count
from collections import defaultdict
from tqdm import tqdm
import json
from pathlib import Path

items = list(drugs_summary_json.items())

smi_to_canon_mol_smi = {}
error_log = defaultdict(list)
stereo_map = defaultdict(set)
n_procs = int(cpu_count() * 0.5)

with Pool(processes=n_procs) as pool:
    for smi, result in tqdm(
        pool.imap_unordered(process_one, items),
        total=len(items),
        desc="Canonicalizing SMILES",
    ):
        if smi is None:
            error_log[result["reason"]].append(result)
            continue

        iso = result["iso"]
        noiso = result["noiso"]

        smi_to_canon_mol_smi[smi] = iso
        stereo_map[noiso].add(iso)

log_dir = Path("script_logs")
log_dir.mkdir(exist_ok=True)
log_path = log_dir / "canon_smiles_failures.json"

with open(log_path, "w") as f:
    json.dump(error_log, f, indent=2)

[04:30:38] WARNING: not removing hydrogen atom without neighbors
[04:30:39] WARNING: not removing hydrogen atom without neighbors
[04:30:42] WARNING: not removing hydrogen atom without neighbors
[04:30:48] Explicit valence for atom # 29 S, 5, is greater than permitted
[04:30:48] Explicit valence for atom # 27 S, 5, is greater than permitted
[04:31:03] WARNING: not removing hydrogen atom without neighbors
[04:31:17] WARNING: not removing hydrogen atom without neighbors
[04:31:20] WARNING: not removing hydrogen atom without neighbors
[04:31:20] WARNING: not removing hydrogen atom without neighbors
[04:31:30] WARNING: not removing hydrogen atom without neighbors
[04:31:32] Explicit valence for atom # 34 S, 5, is greater than permitted
[04:31:43] WARNING: not removing hydrogen atom without neighbors
[04:32:07] WARNING: not removing hydrogen atom without neighbors
[04:32:27] WARNING: not removing hydrogen atom without neighbors
[04:32:28] WARNING: not removing hydrogen atom without neighbor

In [16]:
stereo_map_json = {}
for noiso, iso_set in stereo_map.items():
    if len(iso_set) > 1:
        stereo_map_json[noiso] = list(iso_set)
    
with open("data/rdkit/stereo_map.json", "w") as f:
    json.dump(stereo_map_json, f, indent=2)


In [9]:
new_drugs_summary_json = {}
for smi, data in tqdm(drugs_summary_json.items()):
    if smi in smi_to_canon_mol_smi:
        new_drugs_summary_json[smi] = data

print("New drugs summary JSON length:", len(new_drugs_summary_json))
json.dump(new_drugs_summary_json, open("data/rdkit/new_summary_drugs.json", "w"))


100%|██████████| 304466/304466 [00:00<00:00, 800811.06it/s]


New drugs summary JSON length: 292485


In [ ]:
def murcko_train_val_test_split(smiles, frac_train=0.8, frac_val=0.1):
    from collections import defaultdict
    from rdkit import Chem
    from rdkit.Chem.Scaffolds import MurckoScaffold

    def scaffold(smi):
        mol = Chem.MolFromSmiles(smi)
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)

    scaffold_to_ids = defaultdict(list)
    for i, smi in enumerate(smiles):
        scaffold_to_ids[scaffold(smi)].append(i)

    groups = sorted(scaffold_to_ids.values(), key=len, reverse=True)

    train, val, test = [], [], []
    n_total = len(smiles)
    train_cut = frac_train * n_total
    val_cut = (frac_train + frac_val) * n_total

    count = 0
    for g in groups:
        if count < train_cut:
            train += g
        elif count < val_cut:
            val += g
        else:
            test += g
        count += len(g)

    return train, val, test